# PV Analysis Using Isobaric Surfaces

This notebook is used for debug.

**Key modifications:**
  - **dy:** Since the region (60°N–90°N) has nearly constant meridional spacing, the mean value (a scalar) is used.
  - **dx:** Although the zonal spacing varies with latitude, for each fixed latitude dx is uniform. We compute a 1D array (depending on latitude).
  The key is using dx[none,:] dy[none,:] or dx[:,0] dy[0,:] the funcion can be used. But dx[:,0] dy[0,:] is not correct, it's always the same value which is not true.

In [ ]:
#!/usr/bin/env python3
"""
Debugging Script for potential_vorticity_baroclinic with 1D dx and dy arrays

This script loads model data for year "0008" and month "Feb" from the nocoupl experiment,
averages over ensemble members, extracts one time slice of temperature (T),
zonal wind (U), and meridional wind (V), computes potential temperature (theta),
adds a cyclic point along longitude, rebuilds the horizontal grid, and recomputes the
grid spacings. It then extracts 1D dx and 1D dy arrays (with lengths one less than the
u field along the x and y directions, respectively) and wraps them with units.
These 1D arrays, along with the other inputs, are passed to MetPy's potential_vorticity_baroclinic.
"""

import os
import numpy as np
import xarray as xr
import metpy.calc as mpcalc
from metpy.units import units
from cartopy.util import add_cyclic_point

def load_experiment_field_for_var(file_pattern, var_name, lat_min=60, lat_max=90):
    """
    Load data for a given variable from multiple netCDF files, average over ensemble members,
    and select the specified latitude range.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    data = ds[var_name]
    data = data.sel(lat=slice(lat_min, lat_max))
    if 'member' in data.dims:
        data = data.mean(dim='member')
    return data

def load_year_nocouple_field_var(year, month, var_name, lat_min=60, lat_max=90):
    """
    Load variable data for the 'nocoupl' experiment for a given year and month.
    """
    if month == 'Feb':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    elif month == 'Mar':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

def main():
    # Parameters
    year = "0008"
    month = "Feb"
    var_T = "T"
    var_U = "U"
    var_V = "V"

    print(f"Loading data for year {year}, month {month} (nocoupl experiment)...")
    # Load T, U, V data (averaged over ensemble members)
    T_nocoup = load_year_nocouple_field_var(year, month, var_T)
    U_nocoup = load_year_nocouple_field_var(year, month, var_U)
    V_nocoup = load_year_nocouple_field_var(year, month, var_V)
    print("Data loaded successfully.\n")

    # Extract latitude, longitude, and pressure levels
    lat_vals = T_nocoup.lat.values   # shape: (16,)
    lon_vals = T_nocoup.lon.values   # shape: (144,)
    plev_vals = T_nocoup.plev.values  # shape: (23,)
    print("Array dimensions:")
    print("lat shape:", lat_vals.shape)
    print("lon shape:", lon_vals.shape)
    print("plev shape:", plev_vals.shape)

    # Extract first time slice: (nlev, nlat, nlon) = (23, 16, 144)
    T_slice = T_nocoup.isel(time=0).values
    U_slice = U_nocoup.isel(time=0).values
    V_slice = V_nocoup.isel(time=0).values
    print("\nExtracted time slice shapes:")
    print("T_slice shape:", T_slice.shape)
    print("U_slice shape:", U_slice.shape)
    print("V_slice shape:", V_slice.shape)

    # Convert pressure levels from hPa to Pa
    plev_Pa = plev_vals * 100
    print("\nPressure levels (Pa) shape:", plev_Pa.shape)

    # Compute potential temperature theta: shape (23, 16, 144)
    theta = mpcalc.potential_temperature(plev_Pa[:, None, None] * units.Pa, T_slice * units.K)
    print("\nTheta shape:", theta.shape)

    # --- Add cyclic point along the longitude dimension ---
    # For theta, U, V, and the longitude array
    theta_cyclic, lon_cyclic = add_cyclic_point(theta.magnitude, coord=lon_vals)
    U_cyclic, _ = add_cyclic_point(U_slice, coord=lon_vals)
    V_cyclic, _ = add_cyclic_point(V_slice, coord=lon_vals)
    # Reattach units using Quantity wrapper
    theta_cyclic = units.Quantity(theta_cyclic, theta.units)   # in Kelvin
    U_cyclic = units.Quantity(U_cyclic, "m/s")
    V_cyclic = units.Quantity(V_cyclic, "m/s")
    print("\nAfter adding cyclic point:")
    print("theta_cyclic shape:", theta_cyclic.shape)  # Expected: (23, 16, 145)
    print("Updated lon array shape:", lon_cyclic.shape)   # Expected: (145,)

    # Rebuild 2D horizontal grids using cyclic longitude and original latitudes
    lon2d, lat2d = np.meshgrid(lon_cyclic, lat_vals)
    print("lon2d shape:", lon2d.shape)  # Expected: (16, 145)
    print("lat2d shape:", lat2d.shape)  # Expected: (16, 145)

    # Convert latitudes to radians (function expects dimensionless, interpreted as radians)
    lat2d_rad = np.deg2rad(lat2d)
    print("lat2d_rad shape:", lat2d_rad.shape)

    # Recompute grid spacing using the cyclic grids
    dx_array, dy_array = mpcalc.lat_lon_grid_deltas(lon2d, lat2d)
    print("\nComputed dx_array shape (cyclic):", dx_array.shape)  # Expected: (16, 144)
    print("Computed dy_array shape (cyclic):", dy_array.shape)      # Expected: (15, 145)

    # 根据函数要求，dx 应为 1D 数组
    dx_1d = dx_array[None, :]  
    print(dx_1d)
    dy_1d = dy_array[None, :]  
    print(dy_1d)
    # 包装单位
    dx_1d = units.Quantity(dx_1d, "m")
    dy_1d = units.Quantity(dy_1d, "m")
    print("dx_1d shape:", dx_1d.shape)
    print("dy_1d shape:", dy_1d.shape)

    # 打印最终传入 potential_vorticity_baroclinic 的各输入维度
    print("\nFinal inputs for potential_vorticity_baroclinic:")
    print("theta_cyclic shape:", theta_cyclic.shape)         # (23, 16, 145)
    print("Pressure levels shape:", plev_Pa.shape)             # (23,)
    print("U_cyclic shape:", U_cyclic.shape)                   # (23, 16, 145)
    print("V_cyclic shape:", V_cyclic.shape)                   # (23, 16, 145)
    print("dx_1d shape:", dx_1d.shape)                         # (144,)
    print("dy_1d shape:", dy_1d.shape)                         # (15,)
    print("latitude (lat2d_rad) shape:", lat2d_rad.shape)      # (16, 145)

    try:
        pv_field = mpcalc.potential_vorticity_baroclinic(
            theta_cyclic,
            plev_Pa * units.Pa,
            U_cyclic,
            V_cyclic,
            dx=dx_1d,   # 1D array of length 144 for the x-direction
            dy=dy_1d,   # 1D array of length 15 for the y-direction
            latitude=lat2d_rad,
            x_dim=2,
            y_dim=1,
            vertical_dim=0
        )
        if pv_field.ndim == 3:
            pv_debug = pv_field[0, :, :]
        else:
            pv_debug = pv_field

        print("\nComputed PV field shape:", pv_field.shape)
        print("Selected PV debug field shape:", pv_debug.shape)
    except Exception as e:
        print("Error in computing potential vorticity:", e)

if __name__ == "__main__":
    main()
